In [1]:
# ── Cell 0: HuggingFace login ──────────────────────────────────
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
login(token=user_secrets.get_secret("HF_TOKEN1"))
print("HF login: OK")

HF login: OK


In [2]:
!pip install open-clip-torch transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.6 MB/s eta 0:00:00


In [3]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 35.0 MB/s eta 0:00:00


In [4]:
# ── Cell 2: Imports ────────────────────────────────────────────
import os, re, time, warnings
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModel
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import math, nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
warnings.filterwarnings("ignore")
print("Imports: OK")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Imports: OK


In [5]:
# ── Cell 3: Config ─────────────────────────────────────────────
TRAIN_TXT  = "/kaggle/input/datasets/softhell/med-vqa-2021/data/train/label.txt"
TRAIN_IMGS = "/kaggle/input/datasets/softhell/med-vqa-2021/data/train/images"
VAL_TXT    = "/kaggle/input/datasets/softhell/med-vqa-2021/data/val/label.txt"
VAL_IMGS   = "/kaggle/input/datasets/softhell/med-vqa-2021/data/val/images"
EVAL_CSV   = "/kaggle/working/biomedclip_metrics.csv"
OUTPUT_DIR = "/kaggle/working/biomedclip_mlp"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_ID   = "microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

# Training
LEARNING_RATE = 1e-4
NUM_EPOCHS    = 30       # fast per epoch — can afford more
BATCH_SIZE    = 32
DROPOUT       = 0.3

print(f"Device : {DEVICE}")
print(f"Output : {OUTPUT_DIR}")

Device : cuda
Output : /kaggle/working/biomedclip_mlp


In [6]:
# ── Cell 4: Helpers ────────────────────────────────────────────
def normalise(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def get_synonyms(answer):
    raw   = str(answer).strip()
    parts = [p.strip() for p in raw.split(",")]
    if any(len(p.split()) >= 2 for p in parts):
        return [normalise(raw)]
    return [normalise(p) for p in parts if normalise(p)]

def exact_match_any(pred, answer):
    return int(normalise(pred) in get_synonyms(answer))

print("Helpers: OK")

Helpers: OK


In [7]:
# ── Cell 5: Build answer vocabulary from training set ──────────
#
# BiomedCLIP is a classifier — it needs a fixed answer vocabulary.
# We build this from the training labels only (no val leakage).

train_df = pd.read_csv(TRAIN_TXT, sep="|",
                       names=["image_id", "question", "answer"], header=None)

# Collect all unique normalised answers
all_answers = []
for ans in train_df["answer"]:
    syns = get_synonyms(str(ans))
    if syns:
        all_answers.append(syns[0])

# Build vocab — sorted for reproducibility
vocab      = sorted(set(all_answers))
label2idx  = {label: i for i, label in enumerate(vocab)}
idx2label  = {i: label for label, i in label2idx.items()}
NUM_CLASSES = len(vocab)

print(f"Training vocabulary size : {NUM_CLASSES} unique classes")
print(f"Sample classes           : {vocab[:5]}")

Training vocabulary size : 330 unique classes
Sample classes           : ['abdominal abscess', 'abdominal aortic aneurysm', 'abdominal compartment syndrome', 'aberrant right subclavian artery arsa', 'achalasia']


In [8]:
# ── Cell 6: Dataset ────────────────────────────────────────────
class VQAMedCLIPDataset(Dataset):
    def __init__(self, txt_path, img_dir, label2idx, split="train"):
        df = pd.read_csv(txt_path, sep="|",
                         names=["image_id", "question", "answer"], header=None)
        self.records   = []
        self.label2idx = label2idx
        skipped        = 0

        for _, row in df.iterrows():
            img_id   = str(row["image_id"]).strip()
            fname    = img_id if img_id.endswith((".jpg",".png")) else img_id+".jpg"
            path     = os.path.join(img_dir, fname)
            if not os.path.exists(path):
                continue

            syns = get_synonyms(str(row["answer"]))
            if not syns:
                continue

            primary = syns[0]
            # For training: skip answers not in vocab
            # For val: keep all (OOV gets -1, always wrong)
            label_idx = label2idx.get(primary, -1)
            if split == "train" and label_idx == -1:
                skipped += 1
                continue

            self.records.append({
                "img_path":   path,
                "question":   str(row["question"]).strip(),
                "answer":     primary,
                "raw_answer": str(row["answer"]).strip(),
                "label_idx":  label_idx,
            })

        print(f"  [{split}] Loaded: {len(self.records)} | Skipped OOV: {skipped}")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        return {
            "img_path":   r["img_path"],
            "question":   r["question"],
            "answer":     r["answer"],
            "raw_answer": r["raw_answer"],
            "label_idx":  r["label_idx"],
        }

In [9]:
# ── Cell 7: MLP Classifier head ────────────────────────────────
class BiomedCLIPClassifier(nn.Module):
    def __init__(self, clip_model, num_classes, dropout=0.3):
        super().__init__()
        self.clip     = clip_model
        embed_dim     = 512  # BiomedCLIP output dim
        self.mlp      = nn.Sequential(
            nn.Linear(embed_dim * 2, 1024),
            nn.LayerNorm(1024),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )

    def forward(self, images, texts):
        img_feat = self.clip.encode_image(images)   # (B, 512)
        txt_feat = self.clip.encode_text(texts)     # (B, 512)
        # L2 normalise
        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
        txt_feat = txt_feat / txt_feat.norm(dim=-1, keepdim=True)
        fused    = torch.cat([img_feat, txt_feat], dim=-1)  # (B, 1024)
        return self.mlp(fused)

In [10]:
# ── Cell 8: Load BiomedCLIP with LoRA ────────────────────────────────────
import open_clip
from peft import LoraConfig, get_peft_model
from torchvision import transforms  # <--- Added for augmentation

print("Loading BiomedCLIP model + processor …")
clip_model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
    "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224")

tokenizer = open_clip.get_tokenizer(
    "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224")

# Define Radiology-Safe Training Augmentations
# We chain them directly in front of preprocess_val to keep sizing identical
medical_train_transforms = transforms.Compose([
    transforms.RandomRotation(degrees=10),                       # Safe: minor patient skew
    transforms.ColorJitter(brightness=0.12, contrast=0.12),      # Safe: varying scanner calibrations
    preprocess_val                                               # Final Resize, Crop, & Normalization
])

print("Injecting LoRA adapters into BiomedCLIP …")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["query", "key", "value", "dense", "c_fc", "c_proj"],
    lora_dropout=0.1,
    bias="none",
)

clip_model = get_peft_model(clip_model, lora_config)
clip_model.print_trainable_parameters()
clip_model = clip_model.to(DEVICE)

print("\nBuilding classifier …")
model = BiomedCLIPClassifier(clip_model, NUM_CLASSES, DROPOUT).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params     : {total_params/1e6:.1f}M")
print(f"Trainable params : {trainable_params/1e6:.1f}M")

Loading BiomedCLIP model + processor …


open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

open_clip_pytorch_model.bin:   0%|          | 0.00/784M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Injecting LoRA adapters into BiomedCLIP …
trainable params: 2,654,208 || all params: 198,556,929 || trainable%: 1.3367

Building classifier …
Total params     : 200.3M
Trainable params : 4.4M


In [11]:
# ── Cell 9: DataLoaders ────────────────────────────────────────
class VQAMedCLIPDataset(Dataset):
    def __init__(self, txt_path, img_dir, label2idx, transform=None, split="train"):
        df = pd.read_csv(txt_path, sep="|",
                         names=["image_id", "question", "answer"], header=None)
        self.records   = []
        self.label2idx = label2idx
        self.transform = transform # <--- Added
        skipped        = 0

        for _, row in df.iterrows():
            img_id   = str(row["image_id"]).strip()
            fname    = img_id if img_id.endswith((".jpg",".png")) else img_id+".jpg"
            path     = os.path.join(img_dir, fname)
            if not os.path.exists(path):
                continue

            syns = get_synonyms(str(row["answer"]))
            if not syns:
                continue

            primary = syns[0]
            label_idx = label2idx.get(primary, -1)
            if split == "train" and label_idx == -1:
                skipped += 1
                continue

            self.records.append({
                "img_path":   path,
                "question":   str(row["question"]).strip(),
                "answer":     primary,
                "raw_answer": str(row["answer"]).strip(),
                "label_idx":  label_idx,
            })

        print(f"  [{split}] Loaded: {len(self.records)} | Skipped OOV: {skipped}")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        
        # Load and transform image here so splits get different treatments
        img = Image.open(r["img_path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
            
        return {
            "image":      img,         # <--- Preprocessed Tensor
            "img_path":   r["img_path"], # Keep this for your Cell 13 XAI code!
            "question":   r["question"],
            "answer":     r["answer"],
            "raw_answer": r["raw_answer"],
            "label_idx":  r["label_idx"],
        }

def collate_clip(batch):
    questions   = [s["question"] for s in batch]
    label_idxs  = torch.tensor([s["label_idx"] for s in batch], dtype=torch.long)
    answers     = [s["answer"]     for s in batch]
    raw_answers = [s["raw_answer"] for s in batch]

    # Stack the pre-processed tensors directly from the dataset item
    images = torch.stack([s["image"] for s in batch])
    texts  = tokenizer(questions)

    return {
        "images":     images,
        "texts":      texts,
        "label_idx":  label_idxs,
        "answer":     answers,
        "raw_answer": raw_answers,
    }

print("\nBuilding datasets …")
# Pass the augmentation transforms to train, and standard validation transforms to val
train_dataset = VQAMedCLIPDataset(TRAIN_TXT, TRAIN_IMGS, label2idx, transform=medical_train_transforms, split="train")
val_dataset   = VQAMedCLIPDataset(VAL_TXT,   VAL_IMGS,   label2idx, transform=preprocess_val, split="val")

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                           shuffle=True,  collate_fn=collate_clip,
                           num_workers=2, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                           shuffle=False, collate_fn=collate_clip,
                           num_workers=2, pin_memory=True)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")


Building datasets …
  [train] Loaded: 4500 | Skipped OOV: 0
  [val] Loaded: 500 | Skipped OOV: 0
Train batches : 141
Val batches   : 16


In [12]:
# ── Cell 10: Optimizer + scheduler (LoRA Version) ─────────────────────────

# Train only the parameters that require gradients (LoRA adapters + MLP head)
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=5e-4,             # Slightly higher learning rate for LoRA
    weight_decay=1e-3    # Stronger weight decay for regularization
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS)

criterion    = nn.CrossEntropyLoss(ignore_index=-1)
best_val_acc = 0.0
best_ckpt    = os.path.join(OUTPUT_DIR, "best_model.pt")

print(f"Training LoRA + MLP Head for {NUM_EPOCHS} Epochs...")

for epoch in range(1, NUM_EPOCHS + 1):
    
    # ── Train ──────────────────────────────────────────────────
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [train]"):
        images = batch["images"].to(DEVICE)
        texts  = batch["texts"].to(DEVICE)
        lbl    = batch["label_idx"].to(DEVICE)

        optimizer.zero_grad()
        logits = model(images, texts)
        loss   = criterion(logits, lbl)
        loss.backward()
        
        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], 1.0)
        optimizer.step()

        train_loss    += loss.item()
        preds          = logits.argmax(dim=-1)
        valid          = lbl != -1
        train_correct += (preds[valid] == lbl[valid]).sum().item()
        train_total   += valid.sum().item()

    scheduler.step()
    train_acc = train_correct / max(train_total, 1) * 100

    # ── Validation ─────────────────────────────────────────────
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            images = batch["images"].to(DEVICE)
            texts  = batch["texts"].to(DEVICE)
            lbl    = batch["label_idx"].to(DEVICE)
            
            logits = model(images, texts)
            preds  = logits.argmax(dim=-1)
            valid  = lbl != -1
            val_correct += (preds[valid] == lbl[valid]).sum().item()
            val_total   += valid.sum().item()

    val_acc = val_correct / max(val_total, 1) * 100
    print(f"  Epoch {epoch:2d} — loss: {train_loss/len(train_loader):.4f} | "
          f"train: {train_acc:.2f}% | val: {val_acc:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_ckpt)
        print(f"  ✓ Best → {best_ckpt}  ({best_val_acc:.2f}%)")

print(f"\nTraining complete. Best val acc: {best_val_acc:.2f}%")

Training LoRA + MLP Head for 30 Epochs...


Epoch 1/30 [train]: 100%|██████████| 141/141 [03:30<00:00,  1.49s/it]


  Epoch  1 — loss: 5.3262 | train: 5.87% | val: 10.60%
  ✓ Best → /kaggle/working/biomedclip_mlp/best_model.pt  (10.60%)


Epoch 2/30 [train]: 100%|██████████| 141/141 [03:32<00:00,  1.51s/it]


  Epoch  2 — loss: 4.0043 | train: 14.98% | val: 18.60%
  ✓ Best → /kaggle/working/biomedclip_mlp/best_model.pt  (18.60%)


Epoch 3/30 [train]: 100%|██████████| 141/141 [03:32<00:00,  1.51s/it]


... (51 outputs left out)

Epoch 29/30 [train]: 100%|██████████| 141/141 [03:32<00:00,  1.51s/it]


  Epoch 29 — loss: 0.8051 | train: 79.20% | val: 42.60%


Epoch 30/30 [train]: 100%|██████████| 141/141 [03:32<00:00,  1.51s/it]


  Epoch 30 — loss: 0.7932 | train: 80.11% | val: 42.40%

Training complete. Best val acc: 43.80%


In [13]:
# ── Cell 11: Evaluation loop with Unified Semantic Match & Calibration ──
import os
import re
import math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from transformers import AutoTokenizer, AutoModel

print("Loading best checkpoint …")
model.load_state_dict(torch.load(best_ckpt))
model.eval()

# 1. Centralized Independent Evaluator (Run on CPU to safeguard VRAM & maintain cross-model uniformity)
# 1. Centralized Independent Evaluator (Aligned with Qwen baseline)
print("Initializing Centralized Evaluator (Standalone BiomedBERT)...")
BIOMEDBERT_ID = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract"
BIO_TOKENIZER = AutoTokenizer.from_pretrained(BIOMEDBERT_ID)
BIO_EMBEDDER  = AutoModel.from_pretrained(BIOMEDBERT_ID).to("cpu")
BIO_EMBEDDER.eval()
print(f"✓ Loaded {BIOMEDBERT_ID}\n")

# --- CORE METRIC FUNCTIONS ---

def compute_unified_semantic_match(pred_str, gt_str, threshold=0.85):
    """Unified evaluation function to eliminate contrastive anisotropy bias across models."""
    if not pred_str or not gt_str:
        return 0.0, 0
        
    p_clean = re.sub(r"[^\w\s]", " ", str(pred_str).lower()).strip()
    p_clean = re.sub(r"\s+", " ", p_clean)
    g_clean = re.sub(r"[^\w\s]", " ", str(gt_str).lower()).strip()
    g_clean = re.sub(r"\s+", " ", g_clean)
    
    if p_clean == g_clean:
        return 1.0, 1

    inputs = BIO_TOKENIZER([p_clean, g_clean], padding=True, truncation=True, return_tensors="pt").to("cpu")
    
    with torch.no_grad():
        outputs = BIO_EMBEDDER(**inputs)
        attention_mask = inputs["attention_mask"]
        token_embeddings = outputs[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        embeddings = sum_embeddings / sum_mask
        embeddings = F.normalize(embeddings, p=2, dim=-1)
        
    sim = (embeddings[0] @ embeddings[1].T).item()
    return sim, int(sim >= threshold)

def print_calibration_report(wrong_sim_scores, threshold):
    if not wrong_sim_scores:
        print("No wrong-answer similarity scores collected.")
        return
    arr = np.array(wrong_sim_scores)
    print("\n" + "=" * 57)
    print("  SEMANTIC-MATCH THRESHOLD CALIBRATION CHECK")
    print("=" * 57)
    print(f"  Threshold in use : {threshold:.4f}")
    print(f"  Wrong-sim min    : {arr.min():.4f}")
    print(f"  Wrong-sim mean   : {arr.mean():.4f}")
    print(f"  Wrong-sim p90    : {np.percentile(arr, 90):.4f}")
    print(f"  Wrong-sim p95    : {np.percentile(arr, 95):.4f}")
    print(f"  Wrong-sim max    : {arr.max():.4f}")
    n_leaking = int((arr >= threshold).sum())
    print(f"  Wrong answers passing threshold: {n_leaking}/{len(arr)} ({n_leaking/len(arr)*100:.1f}%)")
    if n_leaking / len(arr) > 0.02:
        print("  ⚠ More than 2% of wrong answers are passing the semantic-match "
              "threshold. Consider raising the threshold.")
    print("=" * 57)

def compute_bleu_ngrams(pred_tokens, gt_tokens, smoothfn):
    refs = [gt_tokens]
    hyp  = pred_tokens
    b1   = sentence_bleu(refs, hyp, weights=(1,0,0,0), smoothing_function=smoothfn)
    b2   = sentence_bleu(refs, hyp, weights=(0,1,0,0), smoothing_function=smoothfn)
    b3   = sentence_bleu(refs, hyp, weights=(0,0,1,0), smoothing_function=smoothfn)
    b4   = sentence_bleu(refs, hyp, weights=(0,0,0,1), smoothing_function=smoothfn)
    cum  = math.exp(
        (math.log(b1+1e-10) + math.log(b2+1e-10) +
         math.log(b3+1e-10) + math.log(b4+1e-10)) / 4)
    return b1, b2, b3, b4, cum

# 2. Setup Evaluation Data
val_df   = pd.read_csv(VAL_TXT, sep="|", names=["image_id","question","answer"], header=None)
smoothie = SmoothingFunction().method1

results          = []
total            = 0
exact_sum        = 0
semantic_sum     = 0
sim_score_sum    = 0.0
bleu1_sum        = 0.0
bleu2_sum        = 0.0
bleu3_sum        = 0.0
bleu4_sum        = 0.0
bleu_cum_sum     = 0.0
wrong_sim_scores = []

# --- MATCHING THE EXCISE BASELINE THRESHOLD FROM QWEN ---
SEMANTIC_THRESHOLD = 0.96

print(f"Evaluating on {len(val_df)} samples …\n")

# 3. Inference Loop
for _, row in tqdm(val_df.iterrows(), total=len(val_df)):
    img_id     = str(row["image_id"]).strip()
    question   = str(row["question"]).strip()
    raw_answer = str(row["answer"]).strip()
    primary_gt = get_synonyms(raw_answer)[0]

    fname    = img_id if img_id.endswith((".jpg",".png")) else img_id+".jpg"
    img_path = os.path.join(VAL_IMGS, fname)
    if not os.path.exists(img_path):
        continue

    # Model specific forward execution remains untouched
    image  = preprocess_val(Image.open(img_path).convert("RGB")).unsqueeze(0).to(DEVICE)
    text   = tokenizer([question]).to(DEVICE)

    with torch.no_grad():
        logits   = model(image, text)
        pred_idx = logits.argmax(dim=-1).item()

    pred = idx2label.get(pred_idx, "unknown")

    # Calculate uniform metrics
    em               = exact_match_any(pred, raw_answer)
    sim_score, sem_m = compute_unified_semantic_match(pred, primary_gt, threshold=SEMANTIC_THRESHOLD)
    
    if em:
        sem_m = 1
        sim_score = max(sim_score, 1.0)
    else:
        wrong_sim_scores.append(sim_score)
        
    b1,b2,b3,b4,bcum = compute_bleu_ngrams(pred.split(), primary_gt.split(), smoothie)

    total        += 1
    exact_sum    += em
    semantic_sum += sem_m
    sim_score_sum += sim_score
    bleu1_sum    += b1
    bleu2_sum    += b2
    bleu3_sum    += b3
    bleu4_sum    += b4
    bleu_cum_sum += bcum

    results.append({
        "Image ID":        img_id,
        "Question":        question,
        "Ground Truth":    raw_answer,
        "Prediction":      pred,
        "Exact Match":     em,
        "Semantic Match":  sem_m,
        "Cosine Sim":      round(sim_score, 4),
        "BLEU-1":          round(b1, 4),
        "BLEU-2":          round(b2, 4),
        "BLEU-3":          round(b3, 4),
        "BLEU-4":          round(b4, 4),
        "Cumulative BLEU": round(bcum, 4),
    })

# 4. Save & Aggregate
results_df = pd.DataFrame(results)
results_df.to_csv(EVAL_CSV, index=False)
print(f"\nResults → {EVAL_CSV}")

avg_b1  = bleu1_sum    / total * 100
avg_b2  = bleu2_sum    / total * 100
avg_b3  = bleu3_sum    / total * 100
avg_b4  = bleu4_sum    / total * 100
avg_em  = exact_sum    / total * 100
avg_sem = semantic_sum / total * 100
avg_sim = sim_score_sum / total

avg_wrong_sim = sum(wrong_sim_scores) / len(wrong_sim_scores) if wrong_sim_scores else 0.0
cumulative_bleu = (avg_b1 * avg_b2 * avg_b3 * avg_b4) ** 0.25

# 5. Mirror Report Output Formatting
print("\n" + "=" * 57)
print("   BIOMEDCLIP + MLP — VQA-MED 2021 RESULTS")
print("=" * 57)
print(f"  Total             : {total}")
print(f"  Exact accuracy    : {avg_em:.2f}%")
print(f"  Semantic accuracy : {avg_sem:.2f}%  (Pass Rate)")
print(f"  Mean Cosine Sim   : {avg_sim:.4f}  (Scale 0-1)")
print(f"  BLEU-1            : {avg_b1:.2f}")
print(f"  BLEU-2            : {avg_b2:.2f}")
print(f"  BLEU-3            : {avg_b3:.2f}")
print(f"  BLEU-4            : {avg_b4:.2f}")
print(f"  Cumulative BLEU   : {cumulative_bleu:.2f}")
print("-" * 57)
print(f"  Avg Sim of Wrong  : {avg_wrong_sim:.4f}")
print("=" * 57)

# Print the calibration report at the very end
print_calibration_report(wrong_sim_scores, threshold=SEMANTIC_THRESHOLD)

Loading best checkpoint …
Initializing Centralized Evaluator (Standalone BiomedBERT)...


pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Loaded microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract



model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Evaluating on 500 samples …




100%|██████████| 500/500 [00:40<00:00, 12.33it/s]


Results → /kaggle/working/biomedclip_metrics.csv

   BIOMEDCLIP + MLP — VQA-MED 2021 RESULTS
  Total             : 500
  Exact accuracy    : 43.80%
  Semantic accuracy : 48.20%  (Pass Rate)
  Mean Cosine Sim   : 0.9550  (Scale 0-1)
  BLEU-1            : 46.41
  BLEU-2            : 35.53
  BLEU-3            : 21.18
  BLEU-4            : 12.09
  Cumulative BLEU   : 25.49
---------------------------------------------------------
  Avg Sim of Wrong  : 0.9200

  SEMANTIC-MATCH THRESHOLD CALIBRATION CHECK
  Threshold in use : 0.9600
  Wrong-sim min    : 0.8126
  Wrong-sim mean   : 0.9200
  Wrong-sim p90    : 0.9562
  Wrong-sim p95    : 0.9661
  Wrong-sim max    : 0.9995
  Wrong answers passing threshold: 22/281 (7.8%)
  ⚠ More than 2% of wrong answers are passing the semantic-match threshold. Consider raising the threshold.


In [14]:
# ── Cell 12: Sample predictions ────────────────────────────────
print("\n── 20 sample predictions ──")
sample = results_df.sample(min(20, len(results_df)), random_state=42)
for _, r in sample.iterrows():
    mark = "✓" if r["Exact Match"] else "✗"
    print(f"  {mark}  GT: {str(r['Ground Truth']):<40}  Pred: {r['Prediction']}")


── 20 sample predictions ──
  ✓  GT: lymphangitic carcinomatosis               Pred: lymphangitic carcinomatosis
  ✗  GT: rheumatoid arthritis                      Pred: ankylosing spondylitis
  ✗  GT: odontoid fracture                         Pred: c spine fracture
  ✗  GT: intraosseous lipoma                       Pred: talocalcaneal coalition
  ✗  GT: pulmonary embolism                        Pred: coarctation of the aorta
  ✓  GT: motor vehicle accidents and instrument failure  Pred: motor vehicle accidents and instrument failure
  ✗  GT: round atelectasis                         Pred: pulmonary sarcoidosis
  ✓  GT: classic bosniak class iii complex renal cyst  Pred: classic bosniak class iii complex renal cyst
  ✗  GT: congenital cystic adenomatoid malformation  Pred: hodgkin s lymphoma
  ✓  GT: granular cell tumor                       Pred: granular cell tumor
  ✓  GT: cavernous hemangioma                      Pred: cavernous hemangioma
  ✗  GT: liver hemangioma                

In [15]:
import cv2
import matplotlib.pyplot as plt

print("Initializing XAI Occlusion Pipeline for BiomedCLIP...")

def compute_biomed_occlusion(model, image_tensor, text_tokens, target_idx, 
                             patch_size=32, stride=16, batch_size=64):
    """
    Computes batched occlusion sensitivity for the BiomedCLIP + MLP model.
    """
    model.eval()
    _, channels, height, width = image_tensor.shape
    
    with torch.no_grad():
        base_logits = model(image_tensor, text_tokens)
        base_prob = F.softmax(base_logits, dim=-1)[0, target_idx].item()
        
    y_coords = list(range(0, height - patch_size + 1, stride))
    x_coords = list(range(0, width - patch_size + 1, stride))
    heatmap_grid = np.zeros((len(y_coords), len(x_coords)))
    
    batch_images = []
    batch_indices = []
    
    for r_idx, y in enumerate(y_coords):
        for c_idx, x in enumerate(x_coords):
            # Apply gray mask (normalized zero roughly equals gray in CLIP space)
            perturbed_img = image_tensor.clone()
            perturbed_img[0, :, y:y+patch_size, x:x+patch_size] = 0.0 
            
            batch_images.append(perturbed_img)
            batch_indices.append((r_idx, c_idx))
            
            if len(batch_images) == batch_size or (y == y_coords[-1] and x == x_coords[-1]):
                b_images = torch.cat(batch_images, dim=0).to(DEVICE)
                b_texts  = text_tokens.expand(len(b_images), -1).to(DEVICE)
                
                with torch.no_grad():
                    logits = model(b_images, b_texts)
                    probs = F.softmax(logits, dim=-1)[:, target_idx].cpu().numpy()
                
                for (r, c), prob in zip(batch_indices, probs):
                    # Record how much the confidence DROPPED
                    heatmap_grid[r, c] = max(0.0, base_prob - prob)
                
                batch_images = []
                batch_indices = []
                
    return heatmap_grid, base_prob

# --- Hallucination Analysis ---
# We define a "Grounded" prediction as one where occluding the most important 
# region drops the model's confidence by at least this threshold (e.g., 10%).
VISUAL_RELIANCE_THRESHOLD = 0.10  

hallucination_results = []
grounded_correct = 0
ungrounded_correct = 0
hallucinated_wrong = 0

print(f"Running XAI on validation set (n={len(val_dataset)})...")

for idx in tqdm(range(len(val_dataset))):
    item = val_dataset[idx]
    
    # Re-prepare exact inputs
    image_tensor = preprocess_val(Image.open(item["img_path"]).convert("RGB")).unsqueeze(0).to(DEVICE)
    question_text = item["question"]
    text_tokens = tokenizer([question_text]).to(DEVICE)
    
    # Get standard prediction to find what the model outputted
    with torch.no_grad():
        logits = model(image_tensor, text_tokens)
        pred_idx = logits.argmax(dim=-1).item()
    
    pred_str = idx2label.get(pred_idx, "unknown")
    gt_str = item["answer"]
    
    # 1. Check Semantic Match
    _, sem_match = compute_unified_semantic_match(pred_str, gt_str, threshold=SEMANTIC_THRESHOLD)
    
    # 2. Run Occlusion on the PREDICTED class
    heatmap, base_conf = compute_biomed_occlusion(model, image_tensor, text_tokens, pred_idx)
    max_drop = heatmap.max()
    
    # 3. Categorize
    if sem_match == 1:
        if max_drop >= VISUAL_RELIANCE_THRESHOLD:
            category = "Grounded Correct"
            grounded_correct += 1
        else:
            category = "Ungrounded Correct"
            ungrounded_correct += 1
    else:
        category = "Hallucinated Wrong"
        hallucinated_wrong += 1
        
    hallucination_results.append({
        "Image ID": item["img_path"].split('/')[-1],
        "Question": question_text,
        "Ground Truth": gt_str,
        "Prediction": pred_str,
        "Base Confidence": round(base_conf, 4),
        "Max Conf Drop": round(max_drop, 4),
        "Taxonomy Category": category
    })

# Save and Output
xai_df = pd.DataFrame(hallucination_results)
xai_df.to_csv(os.path.join(OUTPUT_DIR, "biomedclip_xai_hallucinations.csv"), index=False)

print("\n" + "=" * 57)
print(" UNIFIED HALLUCINATION SUMMARY (BiomedCLIP, n=500)")
print("=" * 57)
print(f"  Grounded Correct   : {grounded_correct}/500 ({grounded_correct/500*100:.1f}%)")
print(f"  Ungrounded Correct : {ungrounded_correct}/500 ({ungrounded_correct/500*100:.1f}%)")
print(f"  Hallucinated Wrong : {hallucinated_wrong}/500 ({hallucinated_wrong/500*100:.1f}%)")
print("=" * 57)

Initializing XAI Occlusion Pipeline for BiomedCLIP...
Running XAI on validation set (n=500)...


100%|██████████| 500/500 [39:23<00:00,  4.73s/it]


 UNIFIED HALLUCINATION SUMMARY (BiomedCLIP, n=500)
  Grounded Correct   : 212/500 (42.4%)
  Ungrounded Correct : 29/500 (5.8%)
  Hallucinated Wrong : 259/500 (51.8%)


In [16]:
import cv2
import matplotlib.pyplot as plt

print("Generating qualitative XAI verification panel...")

# 1. Select one distinct representative from each category
categories = ["Grounded Correct", "Ungrounded Correct", "Hallucinated Wrong"]
selected_examples = {}

for cat in categories:
    matches = xai_df[xai_df["Taxonomy Category"] == cat]
    if not matches.empty:
        # Pull a sample from the middle of the stack for stable representation
        selected_examples[cat] = matches.iloc[len(matches) // 2]

# 2. Setup Plot Grid (3 rows x 2 columns: Original vs. XAI Heatmap)
fig, axes = plt.subplots(3, 2, figsize=(12, 16))
fig.suptitle("BiomedCLIP Hallucination Taxonomy & Visual Grounding Maps", 
             fontsize=16, fontweight="bold", y=0.96)

for idx, cat in enumerate(categories):
    if cat not in selected_examples:
        print(f"Skipping visualization for {cat}: No examples found in dataset.")
        continue
        
    ex = selected_examples[cat]
    img_name = ex["Image ID"]
    img_path = os.path.join(VAL_IMGS, img_name)
    
    # Reload raw image for clean plotting
    orig_img = Image.open(img_path).convert("RGB")
    img_np = np.array(orig_img)
    h, w, _ = img_np.shape
    
    # Recompute the specific heatmap for high resolution mapping
    image_tensor = preprocess_val(orig_img).unsqueeze(0).to(DEVICE)
    question_text = ex["Question"]
    text_tokens = tokenizer([question_text]).to(DEVICE)
    
    # Find index of the predicted token to target the heatmap logic
    with torch.no_grad():
        logits = model(image_tensor, text_tokens)
        pred_idx = logits.argmax(dim=-1).item()
        
    # Run occlusion with smaller stride for smooth visual gradients
    heatmap, _ = compute_biomed_occlusion(
        model, image_tensor, text_tokens, pred_idx, patch_size=24, stride=8, batch_size=128
    )
    
    # Normalize and upscale heatmap to match the original image geometry
    if heatmap.max() > 0:
        heatmap_norm = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
    else:
        heatmap_norm = heatmap
        
    heatmap_resized = cv2.resize(heatmap_norm, (w, h), interpolation=cv2.INTER_CUBIC)
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    
    # Blend the heatmap overlay with the original radiograph
    overlay = cv2.addWeighted(img_np, 0.6, heatmap_color, 0.4, 0)
    
    # --- Column 1: Original Image + Clinical Metadata ---
    axes[idx, 0].imshow(orig_img)
    axes[idx, 0].set_title(f"Category: {cat}", fontsize=13, fontweight="bold", color="darkred" if "Wrong" in cat else "darkgreen")
    axes[idx, 0].axis("off")
    
    metadata_text = (
        f"Q: {question_text}\n"
        f"GT: {ex['Ground Truth']}\n"
        f"Pred: {ex['Prediction']}\n"
        f"Confidence Drop: {ex['Max Conf Drop']:.4f}"
    )
    axes[idx, 0].text(10, h - 20, metadata_text, color="black", fontsize=10,
                      weight="bold", bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"))
    
    
    # --- Column 2: XAI Attention Map Overlay ---
    axes[idx, 1].imshow(overlay)
    axes[idx, 1].set_title(f"Occlusion Sensitivity Map (Target: {ex['Prediction']})",
                            fontsize=11, fontstyle="italic")
    axes[idx, 1].axis("off")



    # axes[idx, 1].imshow(overlay)
    # axes[idx, 1].set_title(f"Occlusion Sensitivity Map (Target: {ex['Prediction']})", fontsize=11, linestyle="italic")
    # axes[idx, 1].axis("off")

# 3. Clean layout structure and write to output folder
plt.tight_layout(rect=[0, 0.03, 1, 0.93])
panel_path = os.path.join(OUTPUT_DIR, "biomedclip_taxonomy_panel.png")
plt.savefig(panel_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"✓ Saved category comparison panel → {panel_path}")

Generating qualitative XAI verification panel...
✓ Saved category comparison panel → /kaggle/working/biomedclip_mlp/biomedclip_taxonomy_panel.png
